##Sending a Test event to the Event Hub

In [0]:
print(1+1)
# Run this in a notebook cell

%pip install azure-eventhub
dbutils.library.restartPython()

2
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.secrets.listScopes()

[SecretScope(name='\\'), SecretScope(name='weather-kv-scope')]

In [0]:
%sh databricks secrets describe-scope --scope weather-kv-scope

The Databricks CLI is only supported for interactive use from the web terminal. We recommend using the web terminal to invoke CLI commands. If you would like to interface with Databricks APIs then we recommend using the Databricks Python SDK.


In [0]:
dbutils.secrets.get(scope="weather-kv-scope", key="eventhub-connection-string")

'[REDACTED]'

In [0]:
conn = dbutils.secrets.get("weather-kv-scope", "eventhub-connection-string")
print(conn.split(";")[0])   # prints only the Endpoint part

Endpoint=sb://weather-streaming-event-hub.servicebus.windows.net/


In [0]:
from azure.eventhub import EventHubProducerClient, EventData

import json

# Event Hub configuration — pulled securely from your Key Vault-backed secret scope
EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(scope="weather-kv-scope", key="eventhub-connection-string")
EVENT_HUB_NAME = "weather-streaming-event-hub"  # replace with your actual Event Hub name

# Initialize the Event Hub producer
producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING,
    eventhub_name=EVENT_HUB_NAME
)

# Function to send events to Event Hub
def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)
    print(f"Sent event: {event}")

# Sample weather event
event = {
    "event_id": 1111,
    "city": "Helsinki",
    "temperature_celsius": 8.5,
    "humidity_percent": 72,
    "condition": "Cloudy",
    "timestamp": "2026-10-05T12:30:00Z"
}

# Send the event
send_event(event)

# Close the producer
producer.close()

Sent event: {'event_id': 1111, 'city': 'Helsinki', 'temperature_celsius': 8.5, 'humidity_percent': 72, 'condition': 'Cloudy', 'timestamp': '2026-10-05T12:30:00Z'}


##API Testing

In [0]:
import requests
import json

# Getting secret value from Key Vault
weatherapikey = dbutils.secrets.get(scope="weather-kv-scope", key="weatherapikey")
location = "Helsinki"  # You can replace with any city name

base_url = "http://api.weatherapi.com/v1"
forecast_url = f"{base_url}/forecast.json"

params = {
    'key': weatherapikey,
    'q': location,
    'days': 3  # optional: number of forecast days
}

response = requests.get(forecast_url, params=params)

if response.status_code == 200:
    forecast_weather = response.json()
    print("Forecast Weather:")
    print(json.dumps(forecast_weather, indent=3))
else:
    print(f"Error: {response.status_code}, {response.text}")



Forecast Weather:
{
   "location": {
      "name": "Helsinki",
      "region": "Southern Finland",
      "country": "Finland",
      "lat": 60.176,
      "lon": 24.934,
      "tz_id": "Europe/Helsinki",
      "localtime_epoch": 1791209065,
      "localtime": "2026-10-05 17:04"
   },
   "current": {
      "last_updated_epoch": 1791208800,
      "last_updated": "2026-10-05 17:00",
      "temp_c": 12.8,
      "temp_f": 55.0,
      "is_day": 1,
      "condition": {
         "text": "Patchy rain nearby",
         "icon": "//cdn.weatherapi.com/weather/64x64/day/176.png",
         "code": 1063
      },
      "wind_mph": 8.5,
      "wind_kph": 13.7,
      "wind_degree": 225,
      "wind_dir": "SW",
      "pressure_mb": 1005.0,
      "pressure_in": 29.68,
      "precip_mm": 0.02,
      "precip_in": 0.0,
      "humidity": 89,
      "cloud": 77,
      "feelslike_c": 10.7,
      "feelslike_f": 51.3,
      "windchill_c": 11.5,
      "windchill_f": 52.7,
      "heatindex_c": 12.8,
      "heatindex_f

In [0]:

import requests
import json

# Getting secret value from Key Vault
weatherapikey = dbutils.secrets.get(scope="weather-kv-scope", key="weatherapikey")
location = "Helsinki"  # You can replace with any city name

base_url = "http://api.weatherapi.com/v1"
current_url = f"{base_url}/current.json"

params = {
    'key': weatherapikey,
    'q': location,
    'days': 3  # optional: number of current days
}

response = requests.get(current_url, params=params)

if response.status_code == 200:
    current_weather = response.json()
    print("current Weather:")
    print(json.dumps(current_weather, indent=3))
else:
    print(f"Error: {response.status_code}, {response.text}")



current Weather:
{
   "location": {
      "name": "Helsinki",
      "region": "Southern Finland",
      "country": "Finland",
      "lat": 60.176,
      "lon": 24.934,
      "tz_id": "Europe/Helsinki",
      "localtime_epoch": 1791209238,
      "localtime": "2026-10-05 17:07"
   },
   "current": {
      "last_updated_epoch": 1791208800,
      "last_updated": "2026-10-05 17:00",
      "temp_c": 12.8,
      "temp_f": 55.0,
      "is_day": 1,
      "condition": {
         "text": "Patchy rain nearby",
         "icon": "//cdn.weatherapi.com/weather/64x64/day/176.png",
         "code": 1063
      },
      "wind_mph": 8.5,
      "wind_kph": 13.7,
      "wind_degree": 225,
      "wind_dir": "SW",
      "pressure_mb": 1005.0,
      "pressure_in": 29.68,
      "precip_mm": 0.02,
      "precip_in": 0.0,
      "humidity": 89,
      "cloud": 77,
      "feelslike_c": 10.7,
      "feelslike_f": 51.3,
      "windchill_c": 11.5,
      "windchill_f": 52.7,
      "heatindex_c": 12.8,
      "heatindex_f"



##Complete Code for getting weather data

In [0]:
import requests
import json

# Function to handle the API response
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

# Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
    current_weather_url = f"{base_url}/current.json"
    params = {
        'key': api_key,
        'q': location,
        'aqi': 'yes'
    }
    response = requests.get(current_weather_url, params=params)
    return handle_response(response)

# Function to get Forecast Data
def get_forecast_weather(base_url, api_key, location, days):
    forecast_url = f"{base_url}/forecast.json"
    params = {
        'key': api_key,
        'q': location,
        'days': days
    }
    response = requests.get(forecast_url, params=params)
    return handle_response(response)

# Function to get Alerts
def get_alerts(base_url, api_key, location):
    alerts_url = f"{base_url}/alerts.json"
    params = {
        'key': api_key,
        'q': location,
        'alerts': 'yes'
    }
    response = requests.get(alerts_url, params=params)
    return handle_response(response)

# Flatten and merge the data
def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime': location_data.get('localtime'),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_in': current.get('pressure_in'),
        'precip_in': current.get('precip_in'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index')
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('desc'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }

    return flattened_data

# Main program
def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1/"
    location = "Chennai"  # You can replace with any city name based on your preference
    weatherapikey = dbutils.secrets.get(scope="weather-kv-scope", key="weatherapikey")

    # Get data from API
    current_weather = get_current_weather(base_url, weatherapikey, location)
    forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
    alerts = get_alerts(base_url, weatherapikey, location)

    # Flatten and merge data
    merged_data = flatten_data(current_weather, forecast_weather, alerts)
    print("Weather Data:", json.dumps(merged_data, indent=3))

# Calling the Main Program
fetch_weather_data()


Weather Data: {
   "name": "Chennai",
   "region": "Tamil Nadu",
   "country": "India",
   "lat": 13.0833,
   "lon": 80.2833,
   "localtime": "2026-10-06 16:42",
   "temp_c": 30.2,
   "is_day": 1,
   "condition_text": "Partly Cloudy",
   "condition_icon": "//cdn.weatherapi.com/weather/64x64/day/116.png",
   "wind_kph": 20.2,
   "wind_degree": 131,
   "wind_dir": "SE",
   "pressure_in": 29.84,
   "precip_in": 0.0,
   "humidity": 65,
   "cloud": 32,
   "feelslike_c": 34.1,
   "uv": 1.1,
   "air_quality": {
      "co": 197.0,
      "no2": 5.2,
      "o3": 133.0,
      "so2": 12.8,
      "pm2_5": 20.4,
      "pm10": 25.1,
      "us-epa-index": 2,
      "gb-defra-index": 2
   },
   "alerts": [],
   "forecast": [
      {
         "date": "2026-10-06",
         "maxtemp_c": 31.0,
         "mintemp_c": 27.8,
         "condition": "Overcast"
      },
      {
         "date": "2026-10-07",
         "maxtemp_c": 30.7,
         "mintemp_c": 27.9,
         "condition": "Sunny"
      },
      {
    

##Sending the complete weather data to event hub

In [0]:
import requests
import json
from azure.eventhub import EventHubProducerClient, EventData

# Event Hub configuration — pulled securely from your Key Vault-backed secret scope
EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(scope="weather-kv-scope", key="eventhub-connection-string")
EVENT_HUB_NAME = "weather-streaming-event-hub"  # replace with your actual Event Hub name

# Initialize the Event Hub producer
producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING,
    eventhub_name=EVENT_HUB_NAME
)

# Function to send events to Event Hub
def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)
    print(f"Sent event: {event}")

# Function to handle the API response
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

# Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
    current_weather_url = f"{base_url}/current.json"
    params = {
        'key': api_key,
        'q': location,
        'aqi': 'yes'
    }
    response = requests.get(current_weather_url, params=params)
    return handle_response(response)

# Function to get Forecast Data
def get_forecast_weather(base_url, api_key, location, days):
    forecast_url = f"{base_url}/forecast.json"
    params = {
        'key': api_key,
        'q': location,
        'days': days
    }
    response = requests.get(forecast_url, params=params)
    return handle_response(response)

# Function to get Alerts
def get_alerts(base_url, api_key, location):
    alerts_url = f"{base_url}/alerts.json"
    params = {
        'key': api_key,
        'q': location,
        'alerts': 'yes'
    }
    response = requests.get(alerts_url, params=params)
    return handle_response(response)

# Flatten and merge the data
def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime': location_data.get('localtime'),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_in': current.get('pressure_in'),
        'precip_in': current.get('precip_in'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index')
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('desc'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }

    return flattened_data

# Main program
def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1/"
    location = "Chennai"  # You can replace with any city name based on your preference
    weatherapikey = dbutils.secrets.get(scope="weather-kv-scope", key="weatherapikey")

    # Get data from API
    current_weather = get_current_weather(base_url, weatherapikey, location)
    forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
    alerts = get_alerts(base_url, weatherapikey, location)

    # Flatten and merge data
    merged_data = flatten_data(current_weather, forecast_weather, alerts)

    #Sending the weather data to Event Hub
    send_event(merged_data)

# Calling the Main Program
fetch_weather_data()


Sent event: {'name': 'Chennai', 'region': 'Tamil Nadu', 'country': 'India', 'lat': 13.0833, 'lon': 80.2833, 'localtime': '2026-10-06 16:54', 'temp_c': 30.2, 'is_day': 1, 'condition_text': 'Partly Cloudy', 'condition_icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'wind_kph': 20.2, 'wind_degree': 131, 'wind_dir': 'SE', 'pressure_in': 29.84, 'precip_in': 0.0, 'humidity': 65, 'cloud': 32, 'feelslike_c': 34.1, 'uv': 1.1, 'air_quality': {'co': 197.0, 'no2': 5.2, 'o3': 133.0, 'so2': 12.8, 'pm2_5': 20.4, 'pm10': 25.1, 'us-epa-index': 2, 'gb-defra-index': 2}, 'alerts': [], 'forecast': [{'date': '2026-10-06', 'maxtemp_c': 31.0, 'mintemp_c': 27.8, 'condition': 'Overcast'}, {'date': '2026-10-07', 'maxtemp_c': 30.7, 'mintemp_c': 27.9, 'condition': 'Sunny'}, {'date': '2026-10-08', 'maxtemp_c': 30.7, 'mintemp_c': 27.6, 'condition': 'Sunny'}]}


##Sending the weather data in streaming fashion

In [0]:
display(spark.sql("SHOW CATALOGS"))

catalog
dbw_weather_straming
samples
system


In [0]:
display(spark.sql("""
SHOW SCHEMAS IN dbw_weather_straming
"""))

databaseName
default
information_schema


In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS
dbw_weather_straming.default.weather_volume
COMMENT 'Volume for weather streaming checkpoints'
""")

DataFrame[]

In [0]:
display(spark.sql("""
SHOW VOLUMES IN dbw_weather_straming.default
"""))

database,volume_name
default,weather_volume


In [0]:
volume_path = "/Volumes/dbw_weather_straming/default/weather_volume"

display(dbutils.fs.ls(volume_path))

[]

In [0]:
dbutils.fs.mkdirs(
    f"{volume_path}/checkpoints/weather_stream"
)

True

In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json
import requests

# Event Hub configuration
EVENT_HUB_NAME = "weather-streaming-event-hub"
eventhub_connection_string = dbutils.secrets.get(scope="weather-kv-scope", key="eventhub-connection-string")
weatherapikey = dbutils.secrets.get(scope="weather-kv-scope", key="weatherapikey")

# Initialize the Event Hub producer
producer = EventHubProducerClient.from_connection_string(conn_str=eventhub_connection_string, eventhub_name=EVENT_HUB_NAME)

# Function to send events to Event Hub
def send_event(eventhub_conn, event):
    producer = EventHubProducerClient.from_connection_string(conn_str=eventhub_conn, eventhub_name=EVENT_HUB_NAME)
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)
    producer.close()

# Function to handle the API response
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

# Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
    current_weather_url = f"{base_url}/current.json"
    params = {'key': api_key, 'q': location, 'aqi': 'yes'}
    response = requests.get(current_weather_url, params=params)
    return handle_response(response)

# Function to get Forecast Data
def get_forecast_weather(base_url, api_key, location, days):
    forecast_url = f"{base_url}/forecast.json"
    params = {'key': api_key, 'q': location, 'days': days}
    response = requests.get(forecast_url, params=params)
    return handle_response(response)

# Function to get Alerts
def get_alerts(base_url, api_key, location):
    alerts_url = f"{base_url}/forecast.json"
    params = {'key': api_key, 'q': location, 'alerts': 'yes'}
    response = requests.get(alerts_url, params=params)
    return handle_response(response)

# Flatten and merge the data
def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime': location_data.get('localtime'),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_in': current.get('pressure_in'),
        'precip_in': current.get('precip_in'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index')
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('desc'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }

    return flattened_data

# Main program to fetch weather data
def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1/"
    location = "Chennai"  # Replace with any city name

    current_weather = get_current_weather(base_url, weatherapikey, location)
    forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
    alerts = get_alerts(base_url, weatherapikey, location)

    merged_data = flatten_data(current_weather, forecast_weather, alerts)
    return merged_data

# Process each streaming batch
def process_batch(batch_df, batch_id):
    try:
        weather_data = fetch_weather_data()  # ✅ no argument
        send_event(eventhub_connection_string, weather_data)
    except Exception as e:
        print(f"Error sending events in batch {batch_id}: {str(e)}")
        raise e



# Set up a streaming source (rate source for testing)
streaming_df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

# Write the streaming data using foreachBatch to send weather data to Event Hub
checkpoint_path = (
    "/Volumes/dbw_weather_straming/default/weather_volume/"
    "checkpoints/weather_stream"
)

query = (
    streaming_df.writeStream
        .foreachBatch(process_batch)
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)  # ✅ Supported on serverless clusters
        .start()
)
query.awaitTermination()




# Close the producer after termination
producer.close()


##Sending the weather data to Event Hub every 30 secs

In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json
import requests

# Event Hub configuration
EVENT_HUB_NAME = "weather-streaming-event-hub"
eventhub_connection_string = dbutils.secrets.get(scope="weather-kv-scope", key="eventhub-connection-string")
weatherapikey = dbutils.secrets.get(scope="weather-kv-scope", key="weatherapikey")

# Initialize the Event Hub producer
producer = EventHubProducerClient.from_connection_string(conn_str=eventhub_connection_string, eventhub_name=EVENT_HUB_NAME)

# Function to send events to Event Hub
def send_event(eventhub_conn, event):
    producer = EventHubProducerClient.from_connection_string(conn_str=eventhub_conn, eventhub_name=EVENT_HUB_NAME)
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)
    producer.close()

# Function to handle the API response
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

# Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
    current_weather_url = f"{base_url}/current.json"
    params = {'key': api_key, 'q': location, 'aqi': 'yes'}
    response = requests.get(current_weather_url, params=params)
    return handle_response(response)

# Function to get Forecast Data
def get_forecast_weather(base_url, api_key, location, days):
    forecast_url = f"{base_url}/forecast.json"
    params = {'key': api_key, 'q': location, 'days': days}
    response = requests.get(forecast_url, params=params)
    return handle_response(response)

# Function to get Alerts
def get_alerts(base_url, api_key, location):
    alerts_url = f"{base_url}/forecast.json"
    params = {'key': api_key, 'q': location, 'alerts': 'yes'}
    response = requests.get(alerts_url, params=params)
    return handle_response(response)

# Flatten and merge the data
def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime': location_data.get('localtime'),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_in': current.get('pressure_in'),
        'precip_in': current.get('precip_in'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index')
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('desc'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }

    return flattened_data

# Main program to fetch weather data
def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1/"
    location = "Chennai"  # Replace with any city name

    current_weather = get_current_weather(base_url, weatherapikey, location)
    forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
    alerts = get_alerts(base_url, weatherapikey, location)

    merged_data = flatten_data(current_weather, forecast_weather, alerts)
    return merged_data

# ⏱️ Flexible throttled streaming batch logic

from datetime import datetime, timedelta

# Set your desired interval (in seconds)
SEND_INTERVAL_SECONDS = 30   # Change this to any value you want

# Initialize last sent time so the first batch sends immediately
last_sent_time = datetime.now() - timedelta(seconds=SEND_INTERVAL_SECONDS)

def process_batch(batch_df, batch_id):
    global last_sent_time
    current_time = datetime.now()

    # Check if enough time has passed since last event
    if (current_time - last_sent_time).total_seconds() >= SEND_INTERVAL_SECONDS:
        weather_data = fetch_weather_data()
        send_event(eventhub_connection_string, weather_data)

        # Update last sent time
        last_sent_time = current_time
        print(f"Event Sent at: {last_sent_time} (Batch {batch_id})")


# Set up a streaming source (rate source for testing)
streaming_df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

# Write the streaming data using foreachBatch to send weather data to Event Hub
checkpoint_path = (
    "/Volumes/dbw_weather_straming/default/weather_volume/"
    "checkpoints/weather_stream"
)

query = (
    streaming_df.writeStream
        .foreachBatch(process_batch)
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)  # ✅ Supported on serverless clusters
        .start()
)
query.awaitTermination()




# Close the producer after termination
producer.close()


In [0]:
query.lastProgress


{
    "id": "9eefec3d-0a1f-472c-926c-bf8247b27b74",
    "runId": "43d61dcf-c4fc-46a4-8cc5-e57892f921fa",
    "name": null,
    "timestamp": "2026-10-06T12:40:04.040Z",
    "batchId": 5,
    "batchDuration": 1578,
    "durationMs": {
        "triggerExecution": 1578,
        "queryPlanning": 89,
        "collectSourceMetrics": 1,
        "getBatch": 0,
        "commitOffsets": 214,
        "addBatch": 1003,
        "latestOffset": 0,
        "walCommit": 412
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "RateStreamV2[rowsPerSecond=1, numPartitions=default]",
            "startOffset": "{\"perPartitionOffset\":[1440,1441,1442,1443,1444,1445,1446,1447]}",
            "endOffset": "{\"perPartitionOffset\":[1488,1489,1490,1483,1484,1485,1486,1487]}",
            "latestOffset": "{\"perPartitionOffset\":[1488,1489,1490,1483,1484,1485,1486,1487]}",
            "numInputRows": 0,
            "inputRowsPerSecond": 0.0,
            "p

In [0]:
query.lastProgress


{
    "id": "9eefec3d-0a1f-472c-926c-bf8247b27b74",
    "runId": "384c4b82-03ef-4ca4-b834-9788258aa23f",
    "name": null,
    "timestamp": "2026-10-06T12:46:19.316Z",
    "batchId": 6,
    "batchDuration": 2595,
    "durationMs": {
        "triggerExecution": 2594,
        "queryPlanning": 108,
        "collectSourceMetrics": 0,
        "getBatch": 0,
        "commitOffsets": 239,
        "addBatch": 1949,
        "latestOffset": 0,
        "walCommit": 464
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "RateStreamV2[rowsPerSecond=1, numPartitions=default]",
            "startOffset": "{\"perPartitionOffset\":[1488,1489,1490,1483,1484,1485,1486,1487]}",
            "endOffset": "{\"perPartitionOffset\":[1864,1865,1858,1859,1860,1861,1862,1863]}",
            "latestOffset": "{\"perPartitionOffset\":[1864,1865,1858,1859,1860,1861,1862,1863]}",
            "numInputRows": 0,
            "inputRowsPerSecond": 0.0,
            "